# Step 4 – Feature Engineering and Scaling for Clustering

**Objectives**

1. Select appropriate clustering features.
2. Transform skewed numerical variables where appropriate.
3. Encode categorical variables.
4. Scale numerical variables.
5. Create a final clustering-ready feature matrix.

**Important:** Because this is clustering, `Default` is excluded from the input features. Otherwise, the known outcome would influence the clusters and defeat the purpose of unsupervised segmentation. `Default` is reserved for evaluating the segments afterwards.

## Step 1 – Select clustering features

Seven features are used. Loan amount is log-transformed before scaling.

| Feature | Treatment |
|---|---|
| revenue_log | Already log-transformed; scale |
| loan_amnt | Apply log1p transformation, then scale |
| fico_n | Retain original scale; scale |
| dti_capped | Retain; scale |
| emp_length | One-hot encode |
| purpose | One-hot encode |
| home_ownership_n | One-hot encode |

**Excluded columns**

- `id`: identifier, not a borrower characteristic.
- `Default`: target/outcome, not used in clustering.
- `dti_n`: duplicates the information in `dti_capped`.
- `dti_extreme`: derived from DTI and potentially redundant.
- `revenue`, `revenue_group`, `fico_group`, `dti_group`, `loan_group`: duplicate or derived information.
- `issue_d`, `issue_year`, `issue_month`: excluded from the initial borrower-profile clustering.
- `experience_c`: near-constant (1,347,656 of 1,347,681 records have the same value).
- `addr_state`: excluded from the initial clustering (many categories and fairness concerns; see Section 9 of the EDA).

## Step 2 – Transform skewed numerical features

The earlier EDA showed:

| Feature | Skewness | Decision |
|---|---|---|
| Loan amount | 0.783 | Log transform |
| Revenue (log) | 0.211 | Already transformed |
| FICO | 1.285 | Retain and scale |
| Capped DTI | 0.472 | Retain and scale |

FICO is positively skewed, but because it is a bounded credit score, its original scale is retained initially rather than applying another transformation.

In [3]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    r"D:\HCL_PROJECTS\Credit Portfolio Segmentation and Risk Pattern Analysis Project-3\step2_outputs\cleaned_credit_portfolio.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1347681, 15)


,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,Default,revenue_log,dti_capped,dti_extreme
0,68407277,Dec-2015,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,0,10.915107,5.91,0
1,68355089,Dec-2015,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,0,11.082158,16.06,0
2,68341763,Dec-2015,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,0,11.170449,13.85,0
3,68476807,Dec-2015,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,0,11.556311,25.37,0
4,68426831,Dec-2015,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,0,10.434145,10.20,0


In [4]:
import numpy as np
import pandas as pd

# Create a working dataset
cluster_df = df.copy()

# Log transformation of loan amount
cluster_df['loan_amnt_log'] = np.log1p(
    cluster_df['loan_amnt']
)

# Check skewness before and after transformation
skewness = pd.DataFrame({
    'Original_Skewness': [
        cluster_df['loan_amnt'].skew()
    ],
    'Transformed_Skewness': [
        cluster_df['loan_amnt_log'].skew()
    ]
}, index=['loan_amnt'])

print(skewness.round(4))

           Original_Skewness  Transformed_Skewness
loan_amnt             0.7832               -0.6533


## Step 3 – Encode categorical variables and scale numerical variables

- `StandardScaler` is applied to the numerical features.
- `OneHotEncoder` is applied to the categorical features.
- `ColumnTransformer` combines both transformations.

The one-hot encoded columns are not standardized, which avoids inflating the influence of rare categories.

**Note:** the 30 binary indicator columns outnumber the 4 scaled numerical columns, so categorical differences may carry substantial weight in K-Means distances. This should be reviewed when evaluating the clusters, for example by comparing results with and without the categorical features.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical features
numerical_features = [
    'revenue_log',
    'loan_amnt_log',
    'fico_n',
    'dti_capped'
]

# Categorical features
categorical_features = [
    'emp_length',
    'purpose',
    'home_ownership_n'
]

# Prepare clustering input
X_cluster = cluster_df[
    numerical_features + categorical_features
].copy()

# Numerical scaling and categorical encoding
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_features
        ),
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        )
    ],
    remainder='drop'
)

# Transform the data
X_cluster_scaled = preprocessor.fit_transform(X_cluster)

print("Original clustering data shape:", X_cluster.shape)

print("Transformed feature matrix shape:",
      X_cluster_scaled.shape)

print("\nMatrix type:")
print(type(X_cluster_scaled))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal encoded feature count:",
      X_cluster_scaled.shape[1])

Original clustering data shape: (1347681, 7)
Transformed feature matrix shape: (1347681, 34)

Matrix type:
<class 'scipy.sparse._csr.csr_matrix'>

Numerical features:
['revenue_log', 'loan_amnt_log', 'fico_n', 'dti_capped']

Categorical features:
['emp_length', 'purpose', 'home_ownership_n']

Total encoded feature count: 34


## Step 4 – Validate numerical scaling

In [6]:
# Extract scaled numerical columns
scaled_numerical = X_cluster_scaled[
    :, :len(numerical_features)
]

# Convert sparse matrix to dense if necessary
if hasattr(scaled_numerical, 'toarray'):
    scaled_numerical = scaled_numerical.toarray()

scaled_numerical = pd.DataFrame(
    scaled_numerical,
    columns=numerical_features
)

print("Scaled numerical feature means:")
print(scaled_numerical.mean().round(4))

print("\nScaled numerical feature standard deviations:")
print(scaled_numerical.std().round(4))

Scaled numerical feature means:
revenue_log      0.0
loan_amnt_log    0.0
fico_n          -0.0
dti_capped      -0.0
dtype: float64

Scaled numerical feature standard deviations:
revenue_log      1.0
loan_amnt_log    1.0
fico_n           1.0
dti_capped       1.0
dtype: float64


## Interpretation of the preprocessing results

### 1. Loan amount transformation

| Metric | Skewness |
|---|---|
| Original loan amount | +0.7832 |
| Log-transformed loan amount | -0.6533 |

- The original loan amount was moderately right-skewed.
- After `log1p`, the distribution became left-skewed.
- Absolute skewness decreased from 0.7832 to 0.6533.

The transformation reduced the absolute skewness, although it reversed the direction, so the transformed distribution should not be described as symmetric. It is reasonable to retain `loan_amnt_log` for the initial clustering dataset, and the change should be documented.

### 2. Categorical encoding

The original seven features became 34 features.

| Feature group | Original columns | Final columns |
|---|---|---|
| Numerical | 4 | 4 |
| Categorical | 3 | 30 |
| Total | 7 | 34 |

The 30 categorical columns are `emp_length` (12), `purpose` (14) and `home_ownership_n` (4). One-hot encoding converted each category into an indicator column.

The result is a SciPy CSR sparse matrix, which is efficient for storing encoded data with many zero values.

### 3. Numerical scaling

| Numerical feature | Mean | Standard deviation |
|---|---|---|
| revenue_log | 0.0000 | 1.0000 |
| loan_amnt_log | 0.0000 | 1.0000 |
| fico_n | 0.0000 | 1.0000 |
| dti_capped | 0.0000 | 1.0000 |

All four numerical features have been standardized with `StandardScaler`. This matters because K-Means uses distance calculations: without scaling, variables with larger numerical ranges would dominate.

### 4. Final preprocessing validation

The next cell checks the feature names, duplicates, exclusion of `Default`, matrix shape, and missing or infinite values. The output shows 34 unique features, `Default` excluded, and no NaN or infinite values.

In [7]:
# Get all transformed feature names
feature_names = preprocessor.get_feature_names_out()

print("Total features:", len(feature_names))

print("\nAll transformed feature names:")
print(feature_names.tolist())

# Check for duplicate feature names
print("\nDuplicate feature names:",
      pd.Index(feature_names).duplicated().sum())

# Verify Default is excluded
print("\nDefault included:",
      'Default' in X_cluster.columns)

# Check matrix dimensions
print("\nFinal matrix shape:", X_cluster_scaled.shape)

# Check for missing or invalid values
print("\nContains NaN:",
      np.isnan(X_cluster_scaled.data).any())

print("Contains infinity:",
      np.isinf(X_cluster_scaled.data).any())

Total features: 34

All transformed feature names:
['num__revenue_log', 'num__loan_amnt_log', 'num__fico_n', 'num__dti_capped', 'cat__emp_length_1 year', 'cat__emp_length_10+ years', 'cat__emp_length_2 years', 'cat__emp_length_3 years', 'cat__emp_length_4 years', 'cat__emp_length_5 years', 'cat__emp_length_6 years', 'cat__emp_length_7 years', 'cat__emp_length_8 years', 'cat__emp_length_9 years', 'cat__emp_length_< 1 year', 'cat__emp_length_NI', 'cat__purpose_car', 'cat__purpose_credit_card', 'cat__purpose_debt_consolidation', 'cat__purpose_educational', 'cat__purpose_home_improvement', 'cat__purpose_house', 'cat__purpose_major_purchase', 'cat__purpose_medical', 'cat__purpose_moving', 'cat__purpose_other', 'cat__purpose_renewable_energy', 'cat__purpose_small_business', 'cat__purpose_vacation', 'cat__purpose_wedding', 'cat__home_ownership_n_MORTGAGE', 'cat__home_ownership_n_OTHER', 'cat__home_ownership_n_OWN', 'cat__home_ownership_n_RENT']

Duplicate feature names: 0

Default included: F

## Step 5 – Project documentation

**Feature Engineering and Scaling**

The dataset underwent feature engineering to prepare borrower attributes for clustering analysis. Log transformations were applied to revenue during EDA and to loan amount during this stage. Four numerical features were standardized using `StandardScaler`, while three categorical features were converted using `OneHotEncoder`. The resulting sparse feature matrix contained 1,347,681 observations and 34 features. The default outcome was excluded to ensure that clustering would be based on borrower and loan characteristics rather than known outcomes.